## Purpose

This lesson teaches you how to avoid re-indexing entire document corpora when only a few documents change. Instead of re-processing 10,000 documents when one is modified, you'll implement SHA-256 checksum-based change detection and update only changed chunks in your vector database. You'll achieve 99.75% time savings and 99.8% cost reduction compared to full re-indexing.

## Concepts Covered

- **Checksum-based change detection**: SHA-256 hashing to identify new, modified, deleted, and unchanged documents
- **Chunk diffing**: Comparing document chunks to detect granular changes
- **Targeted upserts**: Updating only modified vectors in Pinecone/vector databases
- **Deletion handling**: Safely removing old vectors without orphaning data
- **Version snapshots**: Creating rollback points before risky updates
- **Rollback mechanisms**: Recovering from failed updates using saved state

## After Completing This Lesson

You will be able to:
- Run change detection on document corpora to identify what needs updating
- Update vector indexes incrementally, saving time and cost
- Rollback safely when updates fail using version snapshots
- Operate offline in demo mode without API keys (change detection works locally)
- Make informed decisions about when incremental indexing is appropriate vs. full re-indexing

## Context in Track

This is **Level 2, Module 5.1: Incremental Indexing & Updates** in the Production Data Management track. You're learning how to manage evolving data in production RAG systems. This module feeds directly into M5.2 (Batch Processing Optimization) where you'll handle larger-scale updates, and later into the A/B Testing modules (M8.2) where you'll evaluate incremental vs. full indexing strategies with real metrics.

# M5.1: Incremental Indexing & Updates

## Learning Objectives

By the end of this notebook, you will:
- Implement SHA-256 checksum-based change detection
- Update only modified chunks (95% time savings)
- Handle document deletions without orphaning vectors
- Manage index versions for safe rollback
- Debug five common incremental indexing failures
- **Critically: Recognize when full re-indexing is superior**

## Why Incremental Indexing?

**The Problem**: Re-indexing 10,000 documents takes 20+ minutes and costs $50
**The Solution**: Update only changed documents (3-5 seconds, $0.10)

**Performance**:
- 99.75% time reduction
- 99.8% cost reduction
- Minimal resource overhead

In [ ]:
# Setup and imports
import sys
import os
from pathlib import Path

# Import core module
from m5_1_incremental_indexing import (
    ChangeDetector,
    IncrementalIndexer,
    IndexVersionManager,
    simple_chunk_function,
    mock_embedding_function
)
from m5_1_incremental_indexing import config

print("✓ Imports successful")
print(f"✓ Working directory: {os.getcwd()}")

# Check for API keys
has_openai = bool(os.getenv("OPENAI_API_KEY"))
has_pinecone = bool(os.getenv("PINECONE_API_KEY"))

if not has_openai or not has_pinecone:
    print("\n⚠️ No OPENAI_API_KEY/PINECONE_API_KEY – running in demo mode (no upserts).")
    print("   Change detection will work; vector operations will use mock functions.")
else:
    print("\n✓ API keys detected – live mode enabled")
# Expected: Imports successful, shows working directory and mode

Import the core modules for incremental indexing. We'll use ChangeDetector for finding changed files, IncrementalIndexer for updating the vector database, and IndexVersionManager for creating rollback points.

## Section 2: Core Concepts - Change Detection

**Why Checksums Over Timestamps?**

Filesystem timestamps are unreliable:
- Can be modified manually
- May change without content changes (metadata updates)
- Inconsistent across file copies

**SHA-256 Checksums**:
- Deterministic: same content → same hash
- Collision-resistant: different content → different hash
- Fast: ~100MB/s on modern hardware

**State Persistence**:
Store document metadata in JSON:
```json
{
  "doc1.txt": {
    "checksum": "abc123...",
    "chunk_ids": ["doc1__chunk_0", "doc1__chunk_1"],
    "last_updated": "2025-01-07T14:30:22",
    "size_bytes": 2847
  }
}
```

In [ ]:
# Demonstrate checksum calculation
detector = ChangeDetector(state_file="demo_state.json")

# Calculate checksum for example document
test_doc = "example_documents/doc1_vector_databases.txt"
checksum = detector.calculate_checksum(test_doc)

print(f"Document: {test_doc}")
print(f"Checksum: {checksum[:16]}... (truncated)")
print(f"Length: {len(checksum)} characters")
# Expected: Shows SHA-256 hash (64 hex chars)

Calculate a SHA-256 checksum for a sample document. This demonstrates how we generate deterministic hashes to detect content changes.

## Section 3: Implementing Change Detection

The `ChangeDetector` categorizes documents into:
1. **New**: Not in previous state
2. **Modified**: Checksum differs from state
3. **Deleted**: In state but not in current corpus
4. **Unchanged**: Checksum matches state (skip processing)

This categorization enables targeted updates.

In [ ]:
# Detect changes in example documents
from glob import glob

document_paths = glob("example_documents/**/*.txt", recursive=True)
print(f"Found {len(document_paths)} documents\n")

report = detector.detect_changes(document_paths)

print(f"📊 Change Report:")
print(f"  New: {len(report.new)}")
print(f"  Modified: {len(report.modified)}")
print(f"  Deleted: {len(report.deleted)}")
print(f"  Unchanged: {len(report.unchanged)}")
print(f"  Total changed: {report.total_changed}")
# Expected: All 3 docs marked as "new" on first run

Run change detection on all documents in the example directory. The detector compares current checksums against stored state to categorize documents as new, modified, deleted, or unchanged.

## Section 4: Incremental Updates

**Delete-Then-Insert Pattern**:
1. **Delete** old vectors for modified/deleted documents
2. **Process** new/modified documents (chunk → embed)
3. **Insert** new vectors into index
4. **Update** state with new checksums and chunk IDs

This ensures no orphaned vectors remain.

⚠️ **Skipping API calls** (no Pinecone/OpenAI keys configured)
Use mock functions for demonstration.

In [ ]:
# Create mock index for demonstration (no API keys needed)
class MockIndex:
    """Mock Pinecone index for demo without API keys."""
    def __init__(self):
        self.vectors = {}
    
    def delete(self, ids):
        for vid in ids:
            self.vectors.pop(vid, None)
        print(f"  [Mock] Deleted {len(ids)} vectors")
    
    def upsert(self, vectors):
        for vid, emb, meta in vectors:
            self.vectors[vid] = (emb, meta)
        print(f"  [Mock] Upserted {len(vectors)} vectors")

# Initialize mock indexer
mock_index = MockIndex()
indexer = IncrementalIndexer(
    index_client=mock_index,
    change_detector=detector,
    embedding_function=mock_embedding_function,
    chunk_function=simple_chunk_function,
    batch_size=10
)

print("✓ Mock indexer initialized")
# Expected: Mock indexer created (no API calls)

Create a mock vector database for demonstration. This allows the notebook to run without API keys while showing the incremental indexing workflow.

In [ ]:
# Run incremental update
stats = indexer.run_incremental_update(document_paths)

print("\n📈 Update Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

print(f"\n✓ Index now contains {len(mock_index.vectors)} vectors")
# Expected: ~15 vectors (3 docs × ~5 chunks each)

Execute the incremental update. The indexer processes only changed documents, deletes old vectors, and upserts new ones—all tracked in the statistics output.

## Section 5: Version Management & Rollback

**Why Version Snapshots?**
- Enable rollback after failed updates
- Audit trail for production systems
- Recovery from accidental deletions

**Snapshot Strategy**:
- Create snapshot **before** each update
- Keep last 10 versions (configurable)
- Automatic cleanup of old versions
- Timestamp-based IDs for sorting

In [ ]:
# Initialize version manager
version_manager = IndexVersionManager(versions_dir="demo_versions", max_versions=5)

# Create snapshot
snapshot_id = version_manager.create_snapshot("demo_state.json")
print(f"Created snapshot: {snapshot_id}")

# List all snapshots
snapshots = version_manager.list_snapshots()
print(f"\n📦 Available snapshots: {len(snapshots)}")
for sid in snapshots[:3]:  # Show first 3
    print(f"  - {sid}")
# Expected: Shows newly created snapshot

Create a version snapshot of the current state. Snapshots enable rollback if future updates fail, providing a safety net for production systems.

## Section 6: Common Failures & Fixes

### Failure 1: Orphaned Vectors

**Symptom**: Old chunks remain after document update
**Cause**: Delete fails but insert succeeds
**Fix**: Always delete before inserting; track chunk IDs

### Failure 2: Race Conditions

**Symptom**: State corruption with concurrent updates
**Cause**: Multiple processes writing simultaneously
**Fix**: File locking (fcntl) or distributed locks

### Failure 3: Version Conflicts

**Symptom**: Wrong version restored or rollback fails
**Cause**: Snapshot ID confusion
**Fix**: List snapshots before rollback, use timestamps

### Failure 4: False Positive Changes

**Symptom**: Unnecessary re-indexing
**Cause**: Timestamp changes without content changes
**Fix**: Use content checksums (SHA-256), not timestamps

### Failure 5: Partial Update Failures

**Symptom**: Some docs updated, others failed
**Cause**: Exception during batch processing
**Fix**: Create snapshot before update, implement retry logic

In [ ]:
# Simulate Failure 1: Orphaned vectors
print("🔴 Simulating orphaned vector scenario...")

# Intentionally create orphaned vector
test_doc = document_paths[0]
old_chunks = detector.get_chunk_ids(test_doc)
print(f"Old chunks: {len(old_chunks)}")

# If we only insert without deleting, we get orphans
print("⚠️  BAD: Insert without delete → orphaned vectors")
print("✓ GOOD: Delete-then-insert (implemented in IncrementalIndexer)")
# Expected: Shows why delete-then-insert is critical

Demonstrate the orphaned vectors failure scenario. If we insert new vectors without deleting old ones, orphaned data accumulates in the index.

## Section 7: Decision Framework

### ✅ Use Incremental Indexing When:
- Document corpus: **500-50,000 documents**
- Update frequency: **< 5 updates/hour per document**
- Budget: **< $500/month**
- Latency tolerance: **3-5 seconds acceptable**
- No cross-document dependencies

### ❌ Do NOT Use When:
- **Small corpora** (< 500 docs) → Full re-index is fast enough
- **High-frequency updates** (> 5/hour) → Event-driven architecture
- **Cross-document dependencies** → Must re-process all
- **Multiple indexers** → Need distributed locking
- **Sub-second latency** → Streaming pipeline

### Alternative Approaches:

| Scenario | Better Solution | Why |
|----------|----------------|-----|
| < 500 docs | Full re-indexing | Simpler, fast enough |
| > 50K docs | Event-driven ETL | Better scalability |
| Real-time | Streaming pipeline | Lower latency |
| High concurrency | Managed service | Built-in coordination |

In [ ]:
# Decision helper function
def should_use_incremental_indexing(doc_count, update_freq_per_hour, budget_usd):
    """Helper to decide if incremental indexing is appropriate."""
    if doc_count < 500:
        return False, "Corpus too small, use full re-indexing"
    if doc_count > 50000:
        return False, "Corpus too large, use event-driven architecture"
    if update_freq_per_hour > 5:
        return False, "Updates too frequent, use streaming pipeline"
    if budget_usd < 50:
        return True, "Perfect fit for incremental indexing!"
    return True, "Good candidate for incremental indexing"

# Example scenarios
scenarios = [
    (300, 1, 100),      # Small corpus
    (5000, 2, 200),     # Perfect fit
    (100000, 1, 1000),  # Too large
    (1000, 10, 500),    # Too frequent
]

for doc_count, freq, budget in scenarios:
    use, reason = should_use_incremental_indexing(doc_count, freq, budget)
    print(f"{doc_count:6d} docs, {freq} upd/hr, ${budget:4d}/mo → {'✓' if use else '✗'} {reason}")
# Expected: Shows decision logic for 4 scenarios

Evaluate different scenarios using a decision helper function. This shows when incremental indexing is appropriate based on corpus size, update frequency, and budget.

## Section 8: Performance Comparison

### Time Comparison

| Corpus Size | Change Detection | Update (1 doc) | Full Re-index | Speedup |
|------------|------------------|----------------|---------------|---------|
| 100 docs   | < 2s            | 3s             | 30s           | 10x     |
| 1,000 docs | 2-3s            | 3-5s           | 5 min         | 60x     |
| 10,000 docs| 10-15s          | 3-5s           | 20+ min       | 240x    |
| 50,000 docs| 60-90s          | 3-5s           | 2+ hours      | 1440x   |

### Cost Comparison

- **Incremental update**: ~$0.10 (1 doc embedding + vector ops)
- **Full re-index**: ~$50 (10K docs × $0.005/doc)
- **Savings**: 99.8%

### Scaling Characteristics

Change detection becomes bottleneck at 50K+ docs (60-90 seconds).
At this scale, consider:
- Database-backed state (PostgreSQL, DynamoDB)
- Distributed change detection
- Event-driven architecture

In [ ]:
# Performance visualization (text-based)
import time

def simulate_update_time(corpus_size, is_incremental):
    """Simulate update times."""
    if is_incremental:
        return 4.0  # 3-5 seconds
    else:
        # Full re-index scales linearly
        return corpus_size * 0.12  # ~0.12s per doc

corpus_sizes = [100, 1000, 10000, 50000]

print("Performance Comparison (1 document update):\n")
print(f"{'Corpus':>10} | {'Incremental':>12} | {'Full Re-index':>14} | {'Speedup':>8}")
print("-" * 60)

for size in corpus_sizes:
    inc_time = simulate_update_time(size, True)
    full_time = simulate_update_time(size, False)
    speedup = full_time / inc_time
    
    print(f"{size:>10,} | {inc_time:>10.1f}s | {full_time:>12.1f}s | {speedup:>7.0f}x")

# Expected: Shows dramatic speedup as corpus grows

Simulate performance comparisons between incremental and full re-indexing. The speedup grows dramatically as corpus size increases.

## Section 9: Production Deployment

### Monitoring Metrics

Track with Prometheus/CloudWatch:
- `change_detection_duration_seconds` (P95 < 10s)
- `update_success_rate` (> 95%)
- `orphaned_vector_count` (should be 0)
- `state_file_size_bytes` (alert at 100MB)

### Safety Mechanisms

1. **File Locking**: Prevent concurrent state corruption (Unix: fcntl)
2. **Version Snapshots**: Create before each risky update
3. **Atomic Writes**: Temp file + rename pattern
4. **Retry Logic**: Exponential backoff for transient failures
5. **Dead Letter Queue**: Failed updates for manual review

### Deployment Checklist

- [ ] Persistent volume for state file
- [ ] Automated state backups (daily)
- [ ] Monitoring alerts configured
- [ ] Rate limiting enabled
- [ ] Load testing completed
- [ ] Rollback procedure documented
- [ ] Disaster recovery plan ready

In [ ]:
# Example production usage pattern
def production_update_workflow(document_paths):
    """Production-safe update workflow with error handling."""
    
    # 1. Create snapshot before update
    snapshot_id = version_manager.create_snapshot("demo_state.json")
    print(f"✓ Created snapshot: {snapshot_id}")
    
    try:
        # 2. Run incremental update
        stats = indexer.run_incremental_update(document_paths)
        print(f"✓ Update successful: {stats}")
        return True
        
    except Exception as e:
        # 3. Rollback on failure
        print(f"✗ Update failed: {e}")
        version_manager.rollback_to_snapshot(snapshot_id, "demo_state.json")
        print(f"✓ Rolled back to: {snapshot_id}")
        return False

# Demonstrate safe update pattern
success = production_update_workflow(document_paths)
print(f"\nUpdate status: {'✓ Success' if success else '✗ Failed'}")
# Expected: Shows production-safe workflow

Implement a production-safe update workflow. This pattern creates a snapshot before updates and automatically rolls back on failure.

## Section 10: Summary & Next Steps

### Key Takeaways

✅ **Do Use Incremental Indexing**:
- 500-50K document corpora
- Low-frequency updates (<5/hour)
- Budget-conscious deployments
- No cross-document dependencies

❌ **Don't Use Incremental Indexing**:
- Small corpora (<500 docs) - just re-index
- High-frequency updates - use streaming
- Large scale (>50K docs) - event-driven
- Multiple concurrent indexers - need distributed locks

### Critical Learnings

1. **Change Detection**: SHA-256 checksums > timestamps
2. **Delete-Then-Insert**: Prevents orphaned vectors
3. **Version Snapshots**: Enable safe rollback
4. **File Locking**: Prevents concurrent corruption
5. **Know Your Limits**: Not a silver bullet

### Trade-offs Acknowledged

- Change detection latency (10-15s at 10K docs)
- State file grows with corpus size
- Single-node coordination only (file locking)
- Cannot handle cross-document dependencies
- Not suitable for real-time (<1s) updates

### Next Modules

- **M5.2**: Batch Processing Optimization
- **M5.3**: Real-time Streaming Updates  
- **M5.4**: Multi-tenant Index Management

### Resources

- Code: `src/m5_1_incremental_indexing/`
- API: `app.py` (FastAPI REST endpoints)
- Tests: `tests/test_smoke.py`
- Docs: `README.md`

---

**Remember**: Incremental indexing is an optimization, not a requirement. Measure first, optimize second. For small corpora, full re-indexing is often simpler and sufficient.

In [ ]:
# Cleanup demo files (optional)
import shutil

print("🧹 Cleanup demo files...")
cleanup_files = ["demo_state.json", "demo_versions"]

for item in cleanup_files:
    if os.path.exists(item):
        if os.path.isdir(item):
            shutil.rmtree(item)
        else:
            os.remove(item)
        print(f"  Removed: {item}")

print("\n✅ Notebook complete!")
print("Next: Try the REST API (python app.py) or run tests (pytest tests/)")
# Expected: Cleanup confirmation

Clean up temporary demo files created during the notebook execution. This removes state files and version snapshots used for demonstration.